In [16]:
import os

# Il percorso base dove Kaggle monta i dataset
dataset_path = '/kaggle/input'

In [17]:
import os
import torch
import numpy as np # Ci serve per gestire la matrice della maschera
from torch.utils.data import Dataset
from PIL import Image

class MFNetDataset(Dataset):
    def __init__(self, base_path, transform=None):
        # 1. Corretto il nome della variabile (da cartella_base a base_path)
        self.dir_termiche = os.path.join(base_path, "images")
        self.dir_maschere = os.path.join(base_path, "labels")
        
        self.lista_file = sorted(os.listdir(self.dir_termiche))

        # Salviamo la "ricetta" delle trasformazioni
        self.transform = transform

    def __len__(self):
        return len(self.lista_file)

    def __getitem__(self, idx):
        nome_file = self.lista_file[idx]
        
        path_termica = os.path.join(self.dir_termiche, nome_file)
        path_maschera = os.path.join(self.dir_maschere, nome_file)
        
        image_termica = Image.open(path_termica).convert('RGB')
        image_maschera = Image.open(path_maschera)

        if self.transform:
            tensor_termica = self.transform(image_termica)
        else:
            tensor_termica = image_termica
            
        tensor_maschera = torch.as_tensor(np.array(image_maschera), dtype=torch.long)

        return tensor_termica, tensor_maschera

In [18]:
from torchvision import transforms

mia_trasformazione = transforms.ToTensor()

dataset_training = MFNetDataset(base_path="/kaggle/input/datasets/danialqashqai/mfnet-dataset/MFNet-dataset/ir_seg_dataset", transform=mia_trasformazione)

termica, maschera = dataset_training[0]

print("Formato Termica:", type(termica), "- Dimensioni:", termica.shape)
print("Formato Maschera:", type(maschera), "- Dimensioni:", maschera.shape)

Formato Termica: <class 'torch.Tensor'> - Dimensioni: torch.Size([3, 480, 640])
Formato Maschera: <class 'torch.Tensor'> - Dimensioni: torch.Size([480, 640])


In [25]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.conv_block(x)
        return x


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=9):
        super(UNet, self).__init__()
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.down1 = DoubleConv(in_channels, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)
        
        self.bottleneck = DoubleConv(512, 1024)

    # IL DECODER
        
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.up_conv1 = DoubleConv(1024, 512)
        
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.up_conv2 = DoubleConv(512, 256)
        
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.up_conv3 = DoubleConv(256, 128)
        
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.up_conv4 = DoubleConv(128, 64)
        
        self.final_conv = nn.Conv2d(64, num_classes, kernel_size=1)
        

    def forward(self, x):
        # --- 1. L'ENCODER (La discesa) ---
        x1 = self.down1(x)
        p1 = self.pool(x1)
        
        x2 = self.down2(p1)
        p2 = self.pool(x2)
        
        x3 = self.down3(p2)
        p3 = self.pool(x3)
        
        x4 = self.down4(p3)
        p4 = self.pool(x4)
        
        # --- 2. IL BOTTLENECK (Il fondo) ---
        b = self.bottleneck(p4)
        
        # --- 3. IL DECODER (La salita) ---
        
        # Step 1 di risalita
        u1 = self.up1(b)                             # Ingrandiamo il bottleneck
        concat1 = torch.cat([x4, u1], dim=1)         # Uniamo la skip connection (x4) con l'immagine ingrandita (u1)
        d1 = self.up_conv1(concat1)                  # Passiamo tutto nel DoubleConv
        
        # Step 2 di risalita
        u2 = self.up2(d1)
        concat2 = torch.cat([x3, u2], dim=1)
        d2 = self.up_conv2(concat2)
        
        # Step 3 di risalita (Tocca a te!)
        u3 = self.up3(d2)
        concat3 = torch.cat([x2, u3], dim=1)
        d3 = self.up_conv3(concat3)
        
        # Step 4 di risalita (Tocca a te!)
        u4 = self.up4(d3)
        concat4 = torch.cat([x1, u4], dim=1)
        d4 = self.up_conv4(concat4)
        
        # --- 4. OUTPUT FINALE ---
        out = self.final_conv(d4)
        
        return out

In [26]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader


# Diciamo a PyTorch di usare la GPU se disponibile, altrimenti la CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sto usando: {device}")

# Creiamo il modello e lo spediamo sulla GPU
modello = UNet(in_channels=3, num_classes=9).to(device)

# Il DataLoader raggruppa le immagini (es. 4 alla volta) e le mischia (shuffle)
dataloader = DataLoader(dataset_training, batch_size=4, shuffle=True)

# La Funzione di Errore (CrossEntropy è perfetta per le classi intere da 0 a 8)
criterio = nn.CrossEntropyLoss()

# L'Ottimizzatore (Adam è il più famoso e stabile, lr è il 'Learning Rate' ovvero la velocità di apprendimento)
ottimizzatore = optim.Adam(modello.parameters(), lr=0.001)


# --- 2. IL CICLO DI ADDESTRAMENTO ---

epoche = 30 # Quante volte guardiamo TUTTO il dataset
for epoca in range(epoche):
    modello.train() # Mette il modello in modalità "studente"
    
    # enumerate ci permette di avere il numero del batch (batch_idx) e i dati
    for batch_idx, (immagini, maschere_vere) in enumerate(dataloader):
        
        # Spostiamo i dati sulla GPU
        immagini = immagini.to(device)
        maschere_vere = maschere_vere.to(device)
        
        # --- TOCCA A TE: I 5 STEP SACRI ---
        
        # Step 1: Forward (Passa 'immagini' dentro il 'modello')
        predizioni = modello(immagini)
        
        # Step 2: Calcola l'errore (Passa 'predizioni' e 'maschere_vere' dentro il 'criterio')
        loss = criterio(predizioni, maschere_vere)
        
        # Step 3: Azzera i gradienti 
        ottimizzatore.zero_grad()
        
        # Step 4: Backward (Usa il metodo .backward() sulla tua 'loss')
        criterio.backward()
        
        # Step 5: Aggiorna i pesi (Usa il metodo .step() sul tuo 'ottimizzatore')
        ottimizzatore.step()
        
        if batch_idx % 10 == 0:
            print(f"Epoca [{epoca+1}/{epoche}] - Batch {batch_idx} - Loss: {loss.item():.4f}")

Sto usando: cpu


KeyboardInterrupt: 